# 1、HumanInTheLoopMiddleware中间件

## 1.1 举例的过程1：工具调用的中断

In [1]:
import truststore
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from tencentcloud.common import profile

# 从.env文件中加载环境变量
load_dotenv(override=True)
truststore.inject_into_ssl()

# CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
# CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
#
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=CLOSEAI_API_KEY,
#     base_url=CLOSEAI_BASE_URL
# )
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
    profile={
        "max_input_tokens": 1_000_000
    }
)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "发送邮件中断了..."
                },
            },
            description_prefix="中断啦！！"
        ),
    ]
)

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({
    "messages": [HumanMessage(content="请帮我查询今天北京的天气"
                                      "查询今日新闻"
                                      "查看ID为 'sk2131421' 的邮件内容，"
                                      "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
                                      "同时做这四件事")]
},
    config=config
)


rprint(response)

## 举例的过程2：指明工具调用请求决策

In [3]:


weather_decision = {
    "type" : "edit",
    "edited_action" : {
        "name" : "get_weather",
        "args" : {"city" : "上海市","is_forcast" : True},
    }
}


news_decision = {
    "type" : "approve"
}


send_email_decision = {
    "type" : "approve"
}



decisions = {
    "decisions" : []
}

interrupts = response.get("__interrupt__",[])
action_requests = interrupts[0].value["action_requests"]


for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts :
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config = config
    )

    for msg in resumed_response["messages"]:
        msg.pretty_print()

>>> 真的执行发送邮件工具了
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================

好的，我来同时处理这四件事！
Tool Calls:
  get_weather (call_00_dZ7MgquHY7U0knine7Zf4001)
 Call ID: call_00_dZ7MgquHY7U0knine7Zf4001
  Args:
    city: 上海市
    is_forcast: True
  get_news (call_01_MyCozW2jhgAxYc14KLgm2864)
 Call ID: call_01_MyCozW2jhgAxYc14KLgm2864
  Args:
  read_email_tool (call_02_MMpzXE1XLCuwRJS3ZdJ33986)
 Call ID: call_02_MMpzXE1XLCuwRJS3ZdJ33986
  Args:
    email_id: sk2131421
  send_email_tool (call_03_dJM2fHo05V9rB64aBAwE7796)
 Call ID: call_03_dJM2fHo05V9rB64aBAwE7796
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
================================= Tool Message =================================
Name: get_weather

上海市今天天气不错
明天下雨
================================= Tool Message ==========